# SSD Object Detection: VGG16 vs MobileNetV2 Backbone
**Pascal VOC2007 · Google Colab T4**

Steps:
1. Setup repo & dependencies
2. Download VOC2007 dataset
3. Download pretrained VGG16 weights
4. Train MobileNetV2-SSD
5. Evaluate mAP & compare speed/params

In [ ]:
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/amdegroot/ssd.pytorch.git
%cd ssd.pytorch
!pip install -q torch torchvision opencv-python matplotlib numpy tqdm

## 2. Download Pascal VOC2007

In [ ]:
%%bash
mkdir -p data
cd data
wget -q --show-progress http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar
wget -q --show-progress http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar
tar xf VOCtrainval_06-Nov-2007.tar
tar xf VOCtest_06-Nov-2007.tar
echo "VOC2007 extracted."
ls VOCdevkit/VOC2007/

## 3. Download Pretrained VGG16 Weights

In [ ]:
%%bash
mkdir -p weights
cd weights
wget -q --show-progress https://s3.amazonaws.com/amdegroot-models/vgg16_reducedfc.pth
echo "VGG16 weights ready."
ls -lh

## 4. Create MobileNetV2-SSD Backbone

In [ ]:
%%writefile mobilenet_ssd.py
import torch
import torch.nn as nn
import torchvision.models as models


class MobileNetV2SSD(nn.Module):
    def __init__(self, num_classes=21):
        super(MobileNetV2SSD, self).__init__()
        self.num_classes = num_classes

        mobilenet = models.mobilenet_v2(pretrained=True)
        features = mobilenet.features

        self.feature_extractor1 = features[:14]   # stride 16, 96 channels
        self.feature_extractor2 = features[14:]   # stride 32, 1280 channels

        self.extras = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(1280, 256, kernel_size=1),
                nn.ReLU(),
                nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
                nn.ReLU()
            ),
            nn.Sequential(
                nn.Conv2d(512, 128, kernel_size=1),
                nn.ReLU(),
                nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
                nn.ReLU()
            ),
        ])

        # Anchors per location: [4, 6, 6, 6] across 4 feature maps
        self.loc_layers = nn.ModuleList([
            nn.Conv2d(96,   4 * 4,  kernel_size=3, padding=1),
            nn.Conv2d(1280, 6 * 4,  kernel_size=3, padding=1),
            nn.Conv2d(512,  6 * 4,  kernel_size=3, padding=1),
            nn.Conv2d(256,  6 * 4,  kernel_size=3, padding=1),
        ])
        self.cls_layers = nn.ModuleList([
            nn.Conv2d(96,   4 * num_classes, kernel_size=3, padding=1),
            nn.Conv2d(1280, 6 * num_classes, kernel_size=3, padding=1),
            nn.Conv2d(512,  6 * num_classes, kernel_size=3, padding=1),
            nn.Conv2d(256,  6 * num_classes, kernel_size=3, padding=1),
        ])

    def forward(self, x):
        sources = []
        loc, cls = [], []

        x = self.feature_extractor1(x)
        sources.append(x)

        x = self.feature_extractor2(x)
        sources.append(x)

        for layer in self.extras:
            x = layer(x)
            sources.append(x)

        for (src, loc_layer, cls_layer) in zip(sources, self.loc_layers, self.cls_layers):
            loc.append(loc_layer(src).permute(0, 2, 3, 1).contiguous())
            cls.append(cls_layer(src).permute(0, 2, 3, 1).contiguous())

        loc = torch.cat([o.view(o.size(0), -1) for o in loc], 1)
        cls = torch.cat([o.view(o.size(0), -1) for o in cls], 1)

        return loc.view(loc.size(0), -1, 4), cls.view(cls.size(0), -1, self.num_classes)


def build_mobilenet_ssd(num_classes=21):
    return MobileNetV2SSD(num_classes=num_classes)

## 5. Train MobileNetV2-SSD

> Checkpoints saved every 10 epochs to `weights/`. Reduce `BATCH_SIZE` to 8 if you hit OOM.

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from data import VOCDetection, detection_collate
from mobilenet_ssd import build_mobilenet_ssd
from layers.modules import MultiBoxLoss
from utils.augmentations import SSDAugmentation
import time, os

NUM_CLASSES = 21
BATCH_SIZE  = 16
LR          = 1e-3
EPOCHS      = 50
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
VOC_ROOT    = './data/VOCdevkit'

print(f'Training on: {DEVICE}')

dataset = VOCDetection(root=VOC_ROOT, transform=SSDAugmentation(300, (104, 117, 123)))
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     collate_fn=detection_collate, num_workers=2)

net       = build_mobilenet_ssd(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.SGD(net.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40], gamma=0.1)
criterion = MultiBoxLoss(NUM_CLASSES, 0.5, True, 0, True, 3, 0.5, False, DEVICE == 'cuda')

os.makedirs('weights', exist_ok=True)

net.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    t0 = time.time()
    for imgs, targets in loader:
        imgs    = imgs.to(DEVICE)
        targets = [t.to(DEVICE) for t in targets]
        out     = net(imgs)
        optimizer.zero_grad()
        loss_l, loss_c = criterion(out, targets)
        loss = loss_l + loss_c
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1:>2}/{EPOCHS} | Loss: {epoch_loss/len(loader):.4f} | {time.time()-t0:.1f}s")
    if (epoch + 1) % 10 == 0:
        path = f'weights/mobilenet_ssd_epoch{epoch+1}.pth'
        torch.save(net.state_dict(), path)
        print(f'  -> Saved {path}')

print('Training complete.')

## 6. Speed & Parameter Comparison

In [ ]:
import torch, time
from ssd import build_ssd
from mobilenet_ssd import build_mobilenet_ssd

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def benchmark_speed(model, n=100):
    model.eval()
    dummy = torch.randn(1, 3, 300, 300).to(DEVICE)
    with torch.no_grad():
        for _ in range(10):          # warmup
            model(dummy)
        t0 = time.time()
        for _ in range(n):
            model(dummy)
    return (time.time() - t0) / n * 1000  # ms/image

vgg_net = build_ssd('test', 300, 21).to(DEVICE)
vgg_net.load_weights('weights/ssd300_mAP_77.43_v2.pth')

mob_net = build_mobilenet_ssd(21).to(DEVICE)
mob_net.load_state_dict(torch.load('weights/mobilenet_ssd_epoch50.pth', map_location=DEVICE))

print('=== Model Comparison ===')
print(f'VGG16-SSD    | Params: {count_params(vgg_net):.1f}M | Latency: {benchmark_speed(vgg_net):.2f}ms')
print(f'MobileNet-SSD| Params: {count_params(mob_net):.1f}M | Latency: {benchmark_speed(mob_net):.2f}ms')

## 7. mAP Evaluation (VOC2007 test set)

In [ ]:
# VGG16 baseline mAP
!python eval.py --trained_model weights/ssd300_mAP_77.43_v2.pth --voc_root ./data/VOCdevkit

In [ ]:
# MobileNetV2-SSD mAP
!python eval.py --trained_model weights/mobilenet_ssd_epoch50.pth --voc_root ./data/VOCdevkit

## 8. Results Summary

| Model | mAP (VOC2007) | Params | Latency (ms) |
|---|---|---|---|
| VGG16-SSD (paper) | 77.2 | ~26M | — |
| VGG16-SSD (ours) | _fill in_ | ~26M | _measure_ |
| MobileNetV2-SSD (ours) | _fill in_ | ~4–5M | _measure_ |

**Analysis notes:**
- Where does MobileNetV2 degrade most? (Hypothesis: small objects)
- Speedup factor vs accuracy drop tradeoff
- Parameter reduction ratio